# 07. Generation - LLM 답변 생성 (시나리오 B)

## 목표
- 검색된 컨텍스트를 기반으로 LLM이 정확한 답변을 생성
- 질문에서 메타데이터 필터를 자동 추출하여 검색 정확도 향상
- 대화 맥락 유지 (후속 질문 대응)
- 문서에 없는 내용은 모른다고 답변

## 모델
- **gpt-5-mini** (시나리오 B 베이스라인)
- 허용 모델: gpt-5-mini, gpt-5-nano, text-embedding-3-small

In [37]:
import pandas as pd
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
openai_client = OpenAI()

In [38]:
ARTIFACTS_DIR = Path("../../artifacts")
CHROMA_DIR = ARTIFACTS_DIR / "chroma_db"
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-5-mini"
COLLECTION_NAME = "rfp_chunks_v1"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(COLLECTION_NAME)
print(f"컬렉션: {collection.count():,}개 청크, 모델: {LLM_MODEL}")

컬렉션: 13,951개 청크, 모델: gpt-5-mini


## 1. 검색 함수 (06번에서 가져옴)

In [39]:
def embed_query(query: str) -> list[float]:
    response = openai_client.embeddings.create(input=query, model=EMBEDDING_MODEL)
    return response.data[0].embedding


def retrieve(query: str, k: int = 5, where: dict = None, where_document: dict = None) -> list[dict]:
    """벡터 DB에서 유사 청크를 검색한다."""
    query_embedding = embed_query(query)
    kwargs = {"query_embeddings": [query_embedding], "n_results": k}
    if where:
        kwargs["where"] = where
    if where_document:
        kwargs["where_document"] = where_document
    results = collection.query(**kwargs)
    chunks = []
    for i in range(len(results['ids'][0])):
        chunks.append({
            'score': round(1 - results['distances'][0][i], 4),
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
        })
    return chunks

## 2. 질문 분석 — 메타데이터 필터 자동 추출

사용자 질문에서 발주기관, 사업명 등을 감지하여 검색 필터를 자동 생성합니다.
06번에서 확인한 것처럼 메타 필터를 걸면 정확도가 크게 올라갑니다.

In [40]:
# 발주기관 목록 로딩 (퍼지 매칭용)
all_meta = collection.get(limit=1, include=['metadatas'])  # 전체 메타 가져오기 비효율 -> 별도 로딩
chunks_df = pd.read_parquet(Path("../../data/processed/chunks.parquet"))
AGENCY_LIST = chunks_df['발주 기관'].unique().tolist()
PROJECT_LIST = chunks_df['사업명'].unique().tolist()
print(f"발주기관: {len(AGENCY_LIST)}개, 사업: {len(PROJECT_LIST)}개")

발주기관: 87개, 사업: 99개


In [41]:
def extract_filters(query: str, chat_history: list = None) -> dict:
    """질문에서 메타데이터 필터를 추출한다.
    
    규칙:
    - 발주기관명이 1개 감지되면 해당 기관 필터
    - 2개 이상 감지되면 필터 없이 검색 (다문서 비교)
    - '다른', '그 외', '외에' 등 감지 시 이전 필터 해제
    - 위 조건에 해당 안 되면 대화 맥락에서 이전 필터 유지
    """
    # 필터 해제 키워드 감지
    release_keywords = ['다른 기관', '다른 사업', '그 외', '외에', '이외', '말고']
    if any(kw in query for kw in release_keywords):
        return None
    
    # 발주기관 감지 (부분 매칭)
    matched_agencies = []
    for agency in AGENCY_LIST:
        short_name = agency.replace('(주)', '').replace('㈜', '').strip()
        # 3자 이상 부분 매칭
        for part in [short_name, short_name[:6], short_name[:4]]:
            if len(part) >= 3 and part in query:
                if agency not in matched_agencies:
                    matched_agencies.append(agency)
                break
    
    # 2개 이상 기관 감지 → 필터 없이 (다문서 비교)
    if len(matched_agencies) >= 2:
        return None
    
    # 1개 기관 감지 → 필터 적용
    if len(matched_agencies) == 1:
        return {'발주기관': matched_agencies[0]}
    
    # 감지 안 됨 → 대화 맥락에서 이전 필터 유지
    if chat_history:
        for msg in reversed(chat_history):
            if msg.get('filter'):
                return msg['filter']
    
    return None


# 테스트
test_queries = [
    "국민연금공단이 발주한 이러닝시스템 사업 요구사항",
    "기초과학연구원 극저온시스템에서 AI 예측 요구사항",
    "고려대학교 차세대 포털이랑 광주과학기술원 학사 시스템 비교",  # 다문서
    "교육 관련해서 다른 기관이 발주한 사업은?",  # 필터 해제
    "사업 예산이 얼마야",  # 필터 없음
]
for q in test_queries:
    f = extract_filters(q)
    print(f"  {q[:45]:45s} -> {f}")


  국민연금공단이 발주한 이러닝시스템 사업 요구사항                    -> {'발주기관': '국민연금공단'}
  기초과학연구원 극저온시스템에서 AI 예측 요구사항                   -> {'발주기관': '기초과학연구원'}
  고려대학교 차세대 포털이랑 광주과학기술원 학사 시스템 비교              -> None
  교육 관련해서 다른 기관이 발주한 사업은?                       -> None
  사업 예산이 얼마야                                    -> None


## 3. 프롬프트 정의

In [42]:
SYSTEM_PROMPT = """당신은 공공입찰 RFP(제안요청서) 분석 전문가입니다.
사용자의 질문에 대해 제공된 RFP 문서 컨텍스트를 기반으로 정확하게 답변합니다.

## 역할
- 입찰메이트 컨설팅 스타트업의 사내 RAG 시스템
- 컨설턴트가 RFP 핵심 정보를 빠르게 파악하도록 지원

## 답변 원칙
1. 반드시 제공된 컨텍스트에 있는 내용만 답변하세요.
2. 컨텍스트에 없는 내용은 "제공된 문서에서 해당 정보를 찾을 수 없습니다"라고 명확히 답변하세요. 추측하지 마세요.
3. 부분적으로만 확인되는 경우 "문서에서 확인된 내용은 다음과 같으며, [X]에 대한 정보는 포함되어 있지 않습니다"로 구분하세요.

## 답변 형식
- 출처를 [사업명 | 발주기관] 형태로 답변 끝에 명시하세요.
- 표 형태의 정보는 마크다운 표 구조를 유지하세요.
- 요구사항 정리 시 구분(기능/성능/보안 등)과 고유번호를 포함하세요.
- 핵심을 먼저 말하고 세부사항을 이어서 설명하세요.

## 질문 유형별 대응
- **단일 문서 질문**: 해당 문서의 관련 섹션을 구조적으로 정리
- **다문서 비교**: 비교 항목(사업개요, 예산, 기간, 주요 요구사항)을 하나의 표로 나란히 비교. 문서별로 별도의 표를 만들지 말고, 항목을 행으로, 문서를 열로 배치하여 간결하게 작성
- **후속 질문**: 이전 대화 맥락을 참고하되, 현재 컨텍스트 기반으로 답변
- **존재하지 않는 정보**: "찾을 수 없습니다"와 함께 관련된 유사 정보가 있으면 안내"""


def build_context(chunks: list[dict], max_chars: int = 8000) -> str:
    """검색된 청크를 LLM 컨텍스트로 조합한다."""
    parts = []
    total = 0
    for c in chunks:
        source = f"[출처: {c['metadata']['사업명']} | {c['metadata']['발주기관']}]"
        chunk_text = f"{source}\n{c['text']}"
        if total + len(chunk_text) > max_chars:
            break
        parts.append(chunk_text)
        total += len(chunk_text)
    return "\n\n---\n\n".join(parts)


print(f"시스템 프롬프트: {len(SYSTEM_PROMPT)}자")


시스템 프롬프트: 732자


## 4. RAG 파이프라인 통합

In [43]:
def rag_query(
    query: str,
    chat_history: list = None,
    k: int = 5,
    model: str = LLM_MODEL,
) -> dict:
    """질문 -> 필터 추출 -> 검색 -> 답변 생성 전체 파이프라인."""
    start = time.time()
    
    # 1) 필터 추출
    where = extract_filters(query, chat_history)
    
    # 2) 검색
    chunks = retrieve(query, k=k, where=where)
    context = build_context(chunks)
    
    # 3) 메시지 구성
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    # 대화 히스토리 추가 (최근 4턴)
    if chat_history:
        for msg in chat_history[-4:]:
            messages.append({"role": "user", "content": msg["user"]})
            messages.append({"role": "assistant", "content": msg["assistant"]})
    
    user_message = f"""다음 RFP 문서 내용을 참고하여 질문에 답변해주세요.

## 참고 문서
{context}

## 질문
{query}"""
    messages.append({"role": "user", "content": user_message})
    
    # 4) LLM 호출
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=8000,
    )
    
    answer = response.choices[0].message.content or '(응답 없음)'
    elapsed = time.time() - start
    
    return {
        "query": query,
        "answer": answer,
        "filter": where,
        "chunks_used": len(chunks),
        "context_chars": len(context),
        "elapsed_sec": round(elapsed, 1),
        "model": model,
        "tokens": {
            "prompt": response.usage.prompt_tokens,
            "completion": response.usage.completion_tokens,
            "total": response.usage.total_tokens,
        },
    }


def display_answer(result: dict):
    """답변 결과를 정리하여 출력한다."""
    print(f"Q: {result['query']}")
    print(f"필터: {result['filter']}")
    print(f"검색: {result['chunks_used']}개 청크, {result['context_chars']:,}자 컨텍스트")
    print(f"토큰: {result['tokens']['prompt']}+{result['tokens']['completion']}={result['tokens']['total']}")
    print(f"시간: {result['elapsed_sec']}초")
    print(f"\nA:\n{result['answer']}")
    print("=" * 60)


## 5. 단일 문서 질문 테스트 (유형 A)

In [44]:
display_answer(rag_query(
    "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘"
))

Q: 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘
필터: {'발주기관': '국민연금공단'}
검색: 5개 청크, 4,865자 컨텍스트
토큰: 2969+2947=5916
시간: 42.9초

A:
핵심 요약
- 제공된 문서에는 이러닝시스템 운영 용역의 요구사항 구성(구분·고유번호·명칭)이 체계적으로 제시되어 있습니다. 요구사항은 교육운영, 이러닝시스템 운영, 콘텐츠 개발·관리, 유지관리, 개인정보·보안, 프로젝트관리 등으로 분류되며 각 항목에 고유번호가 부여되어 있습니다. 문서에는 또한 정보보안·개인정보보호 준수, 인수인계(공단 보유 프로그램·데이터 이전)와 실물 교재 제공(배송비 포함) 등 운영·실무상 유의사항이 명시되어 있습니다.  
- 문서에서 확인된 요구사항 목록(고유번호 포함)은 아래 표와 같습니다. (표에 있는 명칭은 문서에 기재된 요구사항 명칭을 그대로 옮겼습니다.)

요구사항(구분·고유번호·명칭 및 요약)
| 구분 | 고유번호 | 요구사항 명칭 | 간단설명(문서 기재명 기준) |
| --- | --- | --- | --- |
| 교육운영 (기능 요구사항) | FUR-001 | 국민연금공단 역량 모델 기반의 TRM 제시 | 공단 역량모델 기반 TRM 제시 요구 |
| 교육운영 (기능 요구사항) | FUR-002 | 교육운영 방법 | 교육운영 방법 제시 요구 |
| 교육운영 (기능 요구사항) | FUR-003 | 교육 예상인원 | 교육 예상인원 제시 요구 |
| 교육운영 (기능 요구사항) | FUR-004 | 외부콘텐츠 제시 기준 | 외부콘텐츠 제시 기준 제시 요구 |
| 교육운영 (기능 요구사항) | FUR-005 | 외부콘텐츠 기타 요청사항 | 외부콘텐츠 관련 기타 요청사항 제시 요구 |
| 교육운영 (품질 요구사항) | QUR-001 | 교육콘텐츠 평가 및 품질관리 방안 제시 | 콘텐츠 평가·품질관리 방안 제시 요구 |
| 이러닝시스템 운영 (시스템 기능) | SFR-001 | 이러닝시스템 구축 및 관리 기본방향 | 시스템 구축

In [45]:
display_answer(rag_query(
    "한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 사업이 왜 추진되는지 목적을 알려 줘"
))

Q: 한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 사업이 왜 추진되는지 목적을 알려 줘
필터: None
검색: 5개 청크, 4,531자 컨텍스트
토큰: 3180+1053=4233
시간: 16.5초

A:
핵심: 규제준수와 업무생산성 향상을 위해 정상운전 시 주민선량평가시스템(K-RADAC)을 고도화하는 것이 본 사업의 목적입니다.  
세부목적(문서에 명시된 내용 기준)

1) 규제요건 준수 (목적 1)
   - 제한구역경계에서의 연간선량 준수 여부를 확인할 수 있는 체계 구축 필요.  
   - ICRP60 기반 평가장기 개선 필요.  
   - 액체유출물에 의한 주민피폭선량평가 수행 필요.  

2) 업무생산성 및 의사결정 지원(목적 2)
   - UI 개선을 통한 업무개선 및 신속한 의사결정 환경 구축 필요.  
   - 출력 기능 고도화를 통한 생산성 향상 필요.  

3) 시스템 고도화 및 기능확대(목적 3)
   - 정상운전 시 선량평가 관련 규제 수요에 대응하기 위한 K-RADAC 고도화(추진목표).  
   - 평가장기(Fortran 모듈 및 웹시스템) 개선 및 액체유출물 선량평가 기능(포트란 모듈 및 웹시스템) 추가(사업범위).  

4) 기대효과
   - 규제 수요 대응, 데이터 신뢰성 확보 및 생산성 향상.  

출처: [한국원자력연구원 선량평가시스템 고도화 | 한국원자력연구원]


## 6. 후속 질문 테스트 (유형 C)

대화 맥락을 유지하여 후속 질문에 답변합니다.

In [46]:
# 첫 질문
r1 = rag_query("국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘")
display_answer(r1)

# 대화 히스토리 구성
history = [{
    "user": r1["query"],
    "assistant": r1["answer"],
    "filter": r1["filter"],
}]

Q: 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘
필터: {'발주기관': '국민연금공단'}
검색: 5개 청크, 4,865자 컨텍스트
토큰: 2969+2822=5791
시간: 50.4초

A:
핵심 요약
- 문서에 명시된 요구사항은 크게 교육운영, 이러닝시스템 운영(시스템/학사), 콘텐츠 개발·유지관리, 개인정보·정보보안, 프로젝트 관리(계약·인수인계 등)로 구성되어 있습니다. 각 항목은 고유번호(예: FUR-001, SFR-001 등)로 정리되어 있으며, 문서에는 대부분 요구 명칭만 제시되어 있고 상세 항목(세부기능·성능·수치)은 포함되어 있지 않습니다. 아래 표에서 분류·고유번호·요구사항 명칭을 정리하고, 문서에 명시된 추가 비고(배송료 포함, 산출내역서 작성 지침 등)를 덧붙였습니다.

요구사항(구분·고유번호·요구사항 명칭)
| 구분 | 고유번호 | 요구사항 명칭 | 문서에 명시된 설명 / 비고 |
| --- | ---: | --- | --- |
| 교육운영 (기능) | FUR-001 | 국민연금공단 역량 모델 기반의 TRM 제시 | 요구사항 명칭만 제시되어 있음. 상세내용은 문서에 포함되어 있지 않습니다. |
| 교육운영 (기능) | FUR-002 | 교육운영 방법 | 명칭만 제시. 상세운영방식·지표 등은 문서에 포함되어 있지 않습니다. |
| 교육운영 (기능) | FUR-003 | 교육 예상인원 | 명칭만 제시. 구체 인원수·분배는 문서에 포함되어 있지 않습니다. |
| 교육운영 (기능) | FUR-004 | 외부콘텐츠 제시 기준 | 명칭만 제시. (문서에 외부콘텐츠 관련 일반지침 존재) |
| 교육운영 (기능) | FUR-005 | 외부콘텐츠 기타 요청사항 | 명칭만 제시. |
| 교육운영 (품질) | QUR-001 | 교육콘텐츠 평가 및 품질관리 방안 제시 | 명칭만 제시. |
| 이러닝시스템 운영 (시스템기능) | SFR-001 | 이러닝시스템 구축 및 관리 기본방향 | 명칭만 제시. |
| 이러닝시스템 운영 (시스템기능) |

In [47]:
# 후속 질문 — "콘텐츠 개발 관리" (맥락: 국민연금공단 이러닝)
r2 = rag_query("콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘", chat_history=history)
display_answer(r2)

Q: 콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘
필터: {'발주기관': '국민연금공단'}
검색: 5개 청크, 5,071자 컨텍스트
토큰: 5209+2649=7858
시간: 39.4초

A:
핵심 요약
- 콘텐츠 개발·관리 요구사항은 개발 범위(DER-001), 개발 요건(DER-002), 검사·검수(DER-003), 그리고 유지관리(MAR-001, MAR-002)로 구성되어 있습니다.  
- 문서에는 차시수(기 개발 수정 + 신규 60차시, 부서별 필수 10차시), 동영상 차시 길이(30~40분), 웹·모바일 연동(스마트러닝), 모듈화, 프로토타입→검토→승인 절차, 원본·완성본 제공(PPT 다운로드 지원) 등 구체적 개발·운영 지침이 포함되어 있습니다.  
- 반면 상세 기술스펙(파일 포맷 세부, 접근성 기준, SLA/응답시간 등)이나 예산·일정 등은 문서에 포함되어 있지 않습니다.

요구사항 정리 (구분·고유번호 포함)
| 고유번호 | 구분 | 요구사항 명칭 | 핵심 내용(요약) | 세부사항 |
| --- | --- | --- | --- | --- |
| DER-001 | 개발 요구사항 | 콘텐츠 개발 범위 | 개발 대상 차시 수 및 범위 명시 | 기 개발된 공단 콘텐츠 수정 및 신규 60차시 개발, 부서별 필수교육 콘텐츠 10차시 개발 |
| DER-002 | 개발 요구사항 | 콘텐츠 개발 요건 | 개발 방식·형태·검토·제공 규정 등 | - 개발요청 시 즉시 수행 및 개발컨설팅 지원, 개발계획서 제출, 개발 안내 워크숍 개최<br>- 유지보수·호환성·확장성 고려 개발, 이후 수정·추가·삭제 용이하도록 설계<br>- 동영상 타입으로 차시별 30~40분 내외 개발<br>- 웹 학습과 모바일 학습 연동(스마트러닝) 설계<br>- 주제별 세분화된 모듈로 1개 차시 구성 및 모듈별 게시 가능<br>- 멀티미디어·엔터테인먼트 요소 적용(학습 흥미 유발)·학습자 중심 개발<br>- “나의 학습노트”와 연동 가능하도록 개발<br>- 설계서·스토리보드 제작 

In [48]:
# 후속 질문 — 다른 기관 탐색
history.append({"user": r2["query"], "assistant": r2["answer"], "filter": r2["filter"]})
r3 = rag_query("교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?", chat_history=history)
display_answer(r3)

Q: 교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?
필터: None
검색: 5개 청크, 3,505자 컨텍스트
토큰: 6260+2021=8281
시간: 35.4초

A:
핵심 답변
- 예. 제공된 문서에서 국민연금공단 이외에 교육·학습(교육훈련, 운영자 교육 등) 관련 발주·요구사항이 확인되는 기관은 다음 세 곳입니다: 인천광역시(인천일자리플랫폼 ISP), 한국재정정보원(e나라도움 웹 접근성 컨설팅), 한국교육과정평가원(NCIC 시스템 운영 및 개선). 상세 내용은 아래 표 참조하세요.

| 발주기관 | 사업명(문서표기) | 교육/학습 관련 여부(요약) |
| --- | --- | --- |
| 인천광역시 | 인천일자리플랫폼 정보시스템 구축 ISP 수립용역 | 교육훈련계획 제출 요구: 교육 목적·대상·일정·장소·내용·강사 등 포함. 발주기관 요청 시 관련교육 제공 필요. 배포용 교재 및 소요경비는 사업자가 부담. (문서에 명시) |
| 한국재정정보원 | e나라도움 업무시스템 웹 접근성 컨설팅 | 운영자 교육 요구: 운영자 교육 방법·절차·횟수 구체계획 수립, 교육결과보고서 제출, 운영자 대상 지속 교육 및 가이드라인 제시 요구. (문서에 명시) |
| 한국교육과정평가원 | 국가교육과정정보센터(NCIC) 시스템 운영 및 개선 | 사업명(교육과정정보센터 시스템 운영·개선)으로 교육 관련 시스템 사업임이 확인됨. 단, 제공된 문서 발췌에는 구체적 교육훈련 요구사항(예: 교육계획서 제출 여부 등)은 명시되어 있지 않습니다. 문서에서는 유사분야 사업수행 실적 양식 등이 포함되어 있습니다. (부분확인) |

문서 기반 한계 및 비고
- 문서에서 확인된 내용은 위 표의 항목들뿐입니다. NCIC 문서는 사업명과 실적 관련 서식·보안·비공개 항목 등을 포함하지만, 제공된 발췌본에서는 교육계획의 구체적 요구사항이나 교육횟수·비용 부담 주체 등 상세 내용이 포함되어 있지 않습니다. (문서에서 확인된 내용은 다음과 같으며, NCIC의 교육 관련 상세 요구사항에 대한 정보는

## 7. 다문서 비교 테스트 (유형 B)

In [49]:
display_answer(rag_query(
    "고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래?",
    k=10,  # 두 문서를 모두 가져오기 위해 k 증가
))

Q: 고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래?
필터: None
검색: 10개 청크, 8,022자 컨텍스트
토큰: 5016+4149=9165
시간: 59.9초

A:
핵심 요약  
- 비교 핵심: 고려대학교 사업은 대규모(11,270,000,000원), 24개월, 차세대 통합포털·학사시스템 구축(통합로그인·검색·개인화·연계서비스 등)로 기능·보안·평가 기준이 상세함. 광주과학기술원(GIST) 사업은 소규모(157,300,000원), 5개월, 학사조직 개편과 학위 연계과정에 따른 기능개선 위주로 범위·요구사항이 단순·제한적임.  
(세부 비교 표는 아래 참조)

비교표

| 항목 | 고려대학교 차세대 포털·학사 정보시스템 구축 사업 | 학사시스템 기능개선 사업 |
| --- | ---: | ---: |
| 사업기간 | 계약일로부터 24개월 이내 | 계약체결일로부터 5개월 이내 |
| 사업예산 | 11,270,000,000원 (V.A.T 포함, 3년 분할지급) | 157,300,000원 (V.A.T 포함) |
| 계약/입찰방법 | 제한 경쟁 입찰(협상에 의한 계약) | 공개경쟁입찰(협상에 의한 계약체결) |
| 무상유지보수기간 | 12개월(사업종료일로부터) | 제공된 문서에서 해당 정보를 찾을 수 없습니다. |
| 대금지급방법 | 문서에 분할 지급 일정(3년 분할) 명시(2024~2026 학년도별 지급 비율) | 선금 50% 지급, 잔금은 최종검수 완료시 지급 |
| 공동수급 | 제공된 문서에서 해당 정보를 찾을 수 없습니다. | 공동수급불가 |
| 사업범위(주요기능) | 포털시스템(통합로그인, 통합/지능형 검색, 마이페이지, 공지/알림, 일정관리, 커뮤니티, 게시판, 사용자별 정보서비스, 위젯, 연계서비스(웹메일, 챗봇, 전자결재, 학사/행정 서비스) 등), 응용SW개발 | 학사 조직개편에 따른 시스템 개선, 학위 과정별 연계 시스템 개선(비교적 간략히 기재) |
| 추진배경/목적 | 교육환경 변화 대응, 

## 8. 무응답 테스트 (유형 D)

문서에 없는 내용에 대해 모른다고 답변하는지 확인합니다.

In [50]:
display_answer(rag_query(
    "국민연금공단 이러닝시스템 사업의 낙찰 업체가 어디야?"
))

Q: 국민연금공단 이러닝시스템 사업의 낙찰 업체가 어디야?
필터: {'발주기관': '국민연금공단'}
검색: 5개 청크, 3,916자 컨텍스트
토큰: 2924+568=3492
시간: 10.7초

A:
핵심: 제공된 문서에서 낙찰 업체 정보를 찾을 수 없습니다.

세부: 제공된 자료에는 입찰참가신청서 양식, 서식(업체현황·프로젝트 일정 등), 입찰보증금 및 부속서류 안내 등 RFP 관련 양식과 안내문이 포함되어 있으나, 낙찰자(낙찰 업체) 명시는 문서에 없습니다.

[2024년 이러닝시스템 운영 용역 | 국민연금공단]


In [51]:
display_answer(rag_query(
    "2025년 서울시 AI 챗봇 구축 사업에 대해 알려줘"
))

Q: 2025년 서울시 AI 챗봇 구축 사업에 대해 알려줘
필터: {'발주기관': '2025 구미 아시아육상경기선수권대회 조직위원회'}
검색: 5개 청크, 5,160자 컨텍스트
토큰: 3323+1834=5157
시간: 30.7초

A:
핵심: 제공된 문서에서는 "2025년 서울시 AI 챗봇 구축 사업"에 대한 정보가 없습니다. 제시하신 문서에는 구미아시아육상경기선수권대회 종합정보시스템 관련 요구사항만 포함되어 있습니다. (추측하지 않습니다.)

문서에서 확인된 유사·참고 가능한 요구사항(구미 대회 종합정보시스템 RFP에서 발췌)
- 문서에서 확인된 내용은 다음과 같으며, '서울시 AI 챗봇 사업'에 대한 정보는 포함되어 있지 않습니다.

아래는 RFP에 명시된 항목을 분류(구분)하고 고유번호를 붙여 정리한 요약입니다.

| 고유번호 | 구분 | 요구사항 요약 |
| --- | --- | --- |
| QUR-002 | 품질 | - 품질관리 조직과 운영절차를 구체적으로 제시하고 준수할 것<br>- 품질/성능 문제 발생 시 부하 테스트 및 분석결과 제시·개선할 것<br>- 사업 범위 외 요인(운영 서버 등)으로 영향 예상 시 원인 및 해결방안 제시·협조할 것<br>- DB 구축 포함 시 사업완료(검수) 시 데이터 품질진단 및 오류데이터 개선 실시 |
| PERF-001 | 성능 | - 시스템 성능을 고려한 개발 방안 제시 및 안정적 운영 지원 방안 제시<br>- 개발 중 성능 상태 모니터링으로 사전 문제 파악·조치<br>- 기존 장비 활용 및 기술지원·가이드 제공<br>- 사용자 입력 오류 메시지는 정보 입력 후 3초 이내 표시(중요정보는 노출 금지) |
| SWINFO-001 | 준수/제출 | - 「소프트웨어진흥법 제58조」에 따라 SW사업정보 데이터 작성·제출 필요<br>- SW사업정보 관련 제출사항은 사업수행계획서에 단계별 산출물 리스트로 명시<br>- 기능점수 데이터 작성을 위해 기능점수 측정 전문가 포함 필요 |
| DB-001 | 데이터/DB | - 외

## 9. 모호한 질의 테스트 (유형 E)

In [52]:
display_answer(rag_query(
    "기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?"
))

Q: 기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?
필터: {'발주기관': '기초과학연구원'}
검색: 5개 청크, 5,117자 컨텍스트
토큰: 3164+1051=4215
시간: 16.1초

A:
핵심 결론
- 제공된 문서에서 AI 기반 예측(예: 예측모델·머신러닝을 이용한 고장예측·예지보수 등)에 대한 요구사항은 확인되지 않습니다. "제공된 문서에서 해당 정보를 찾을 수 없습니다."

문서에서 확인되는 관련(비-AI) 요구사항(기능 관점)
1. 알람 메시지 확인(표 9 항목 1) — 제어시스템의 알람 모니터링 기능 수행
2. 주요 센서 트렌드 확인(표 9 항목 2) — 온도·압력·유량·속도 등 트렌드 관찰
3. 정상 상황의 주요 센서 지시값 파악 및 숙지(표 9 항목 3)
4. 비정상 상황 감지 및 연구원 통보(표 9 항목 4~5) — 무전통신 활용
5. 제어로직 정상 작동 여부 확인(표 9 항목 6)
6. 제어시스템(알람시스템 포함) 개선·보완에 대한 연구원 협조 요청 가능(문서 일부)

해석 및 구분
- 문서에는 시스템 모니터링·알람 확인·트렌드 관찰 등 운전·감시 기능 요구는 상세히 기술되어 있으나, 이들 기능을 AI(예: 예측 알고리즘)로 구현하라는 명시적 요구나 규격은 포함되어 있지 않습니다.
- 문서에 AI 도입을 전제로 한 성능 기준(예: 예측 정확도, 학습데이터 제공·보안 등) 또한 제시되어 있지 않습니다.

출처: [2025년도 중이온가속기용 극저온시스템 운전 용역 | 기초과학연구원]


In [53]:
# 모니터링 관련 후속 질문
r_cryo = rag_query("기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?")
history_cryo = [{"user": r_cryo["query"], "assistant": r_cryo["answer"], "filter": r_cryo["filter"]}]

display_answer(rag_query(
    "그럼 모니터링 업무에 대한 요청사항이 있는지 찾아보고 알려 줘",
    chat_history=history_cryo,
))

Q: 그럼 모니터링 업무에 대한 요청사항이 있는지 찾아보고 알려 줘
필터: {'발주기관': '기초과학연구원'}
검색: 5개 청크, 5,124자 컨텍스트
토큰: 3833+2450=6283
시간: 32.6초

A:
핵심: 문서에는 모니터링(휴지기간·가동기간) 수행 항목·절차·요구사항이 명확히 기술되어 있습니다. 주요 모니터링 업무 항목, 기록·분석 요구, 사용 시스템·장비 제공 및 계약자 준비물, 교육 요구 등이 포함되어 있습니다. 단, 특정 수치의 기록 간격·분석 방법·결과물 형식 등은 연구원이 별도로 정한다고 명시되어 있어 문서에 구체값은 없습니다.

모니터링 업무(문서에서 확인된 항목)
| 순번 | 모니터링 업무 | 비고 |
|---:|---|---|
| 1 | 알람 메시지 확인 |  |
| 2 | 주요 센서 트렌드 확인 | 온도, 압력, 유량, 속도 등 |
| 3 | 정상 상황의(운전 시) 주요 센서 지시값 파악 및 숙지 | 온도, 압력, 유량, 속도 등 |
| 4 | 비정상 상황 감지 | 연구원에서 현장 확인 필요 시 무전 통신 활용 |
| 5 | (연구원 지시) 정하는 값 관찰·확인 및 상황 전달 | 연구원에서 무전기 제공 |
| 6 | 제어로직 정상 작동 여부 확인 | 자동 밸브는 제어시스템 활용 등 |
| 7 | 연구원 요청 시 현장 상황 일시적/지속적 관찰 | 유지보수 활동 내용·진행 상황 파악 포함, 비방사선구역 한정 |
| 8 | 기타 운전 상황 확인에 필요한 모니터링 | 주기적 현장 순찰 포함 |

요구사항 정리 (구분·고유번호 포함)
- R-01 (기능) 모니터링 항목 수행
  - 계약자는 표에 명시된 알람 확인, 센서 트렌드 확인, 정상 지시값 숙지, 비정상 감지 등 모니터링 업무를 수행해야 함.
- R-02 (운영범위/조건) 모니터링 시점/영역
  - 휴지기간 중 지속적 관찰(유지보수 전·후) 및 가동기간 중 모니터링을 수행. 연구원 요청 시 현장 일시적 또는 지속적 관찰 수행(비방사선구역).
- R-03 (데이터 기록/분석) 수치 기록

## 10. 비용 분석

In [54]:
# gpt-5-mini 가격 기준 (추정)
# input: $0.15/1M tokens, output: $0.60/1M tokens
est_input_price = 0.15 / 1_000_000
est_output_price = 0.60 / 1_000_000

print("=== 질문 1건당 비용 추정 ===")
print(f"  평균 입력 토큰: ~3,000 (시스템+컨텍스트+질문)")
print(f"  평균 출력 토큰: ~500")
print(f"  예상 비용: ${3000*est_input_price + 500*est_output_price:.4f} (약 {(3000*est_input_price + 500*est_output_price)*1350:.1f}원)")
print(f"\n  100건 질문: 약 {(3000*est_input_price + 500*est_output_price)*100*1350:.0f}원")
print(f"  1000건 질문: 약 {(3000*est_input_price + 500*est_output_price)*1000*1350:.0f}원")

=== 질문 1건당 비용 추정 ===
  평균 입력 토큰: ~3,000 (시스템+컨텍스트+질문)
  평균 출력 토큰: ~500
  예상 비용: $0.0008 (약 1.0원)

  100건 질문: 약 101원
  1000건 질문: 약 1012원


## 11. 종합 요약 및 인사이트

### Generation 결과
- gpt-5-mini로 RAG 답변 생성 파이프라인 완성
- 질문 -> 필터 자동 추출 -> 검색 -> 답변 전체 흐름 구현
- 대화 맥락 유지 (후속 질문 대응)

### 왜 이렇게 구현했는가

**질문에서 메타데이터 필터를 자동 추출하는 이유**

06번에서 확인한 것처럼, 필터 없는 벡터 검색은 Top-1 정확도 25%이지만
발주기관 필터를 걸면 거의 모든 질문에서 정답을 찾았다.
사용자가 "국민연금공단"을 언급하면 자동으로 필터를 걸어 검색 정확도를 높인다.

**대화 히스토리에서 이전 필터를 유지하는 이유**

후속 질문 "콘텐츠 개발 관리 요구사항 알려줘"에는 발주기관 정보가 없다.
이전 대화에서 "국민연금공단"이 필터로 사용됐다면 그 맥락을 유지해야
같은 문서 내에서 후속 정보를 찾을 수 있다.

**temperature를 설정하지 않은 이유**

gpt-5-mini는 temperature 파라미터를 지원하지 않아 기본값(1)으로 동작한다.
대신 시스템 프롬프트에서 '컨텍스트에 있는 내용만 답변'을 명시하여 할루시네이션을 억제한다.

**시스템 프롬프트에 '모른다고 답변' 규칙을 명시한 이유**

평가 기준 중 '문서에 없는 내용은 모른다고 답변하는지'가 있으므로,
프롬프트에서 명시적으로 지시해야 LLM이 일관되게 동작한다.

### 다음 단계
- `08_evaluation.ipynb`: 질문 세트 기반 정량 평가
- 평가 결과 기반으로 프롬프트 최적화, 검색 고도화 방향 결정